In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
train_raw = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test.csv')
meal = pd.read_csv('data/meal_info.csv')
centerinfo = pd.read_csv('data/fulfilment_center_info.csv')

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
train = pd.merge(train_raw, meal, on="meal_id", how="left")
df = pd.merge(train, centerinfo, on="center_id", how="left")
print("Shape of train data : ", df.shape)
df.head()

Shape of train data :  (456548, 15)


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
test_raw = pd.merge(test_raw, meal, on="meal_id", how="left")
dft = pd.merge(test_raw, centerinfo, on="center_id", how="left")
print("Shape of train data : ", dft.shape)
dft.head()

Shape of train data :  (32573, 14)


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,category,cuisine,city_code,region_code,center_type,op_area
0,1028232,146,55,1885,158.11,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0
1,1127204,146,55,1993,160.11,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0
2,1212707,146,55,2539,157.14,159.14,0,0,Beverages,Thai,647,56,TYPE_C,2.0
3,1082698,146,55,2631,162.02,162.02,0,0,Beverages,Indian,647,56,TYPE_C,2.0
4,1400926,146,55,1248,163.93,163.93,0,0,Beverages,Indian,647,56,TYPE_C,2.0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
col_names=['center_id','meal_id','category','cuisine','city_code','region_code','center_type','emailer_for_promotion','homepage_featured', 'week']
dft[col_names] = dft[col_names].astype('category')

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
col_names=['center_id','meal_id','category','cuisine','city_code','region_code','center_type','emailer_for_promotion','homepage_featured', 'week']
df[col_names] = df[col_names].astype('category')

print("Train Datatype\n",df.dtypes)

Train Datatype
 id                          int64
week                     category
center_id                category
meal_id                  category
checkout_price            float64
base_price                float64
emailer_for_promotion    category
homepage_featured        category
num_orders                  int64
category                 category
cuisine                  category
city_code                category
region_code              category
center_type              category
op_area                   float64
dtype: object


In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
df = df[df['num_orders'] <= 20000];
df=df.drop("id",  axis=1)

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
if 'id' in df.columns:
    df = df.drop('id', axis=1)
df.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
df['new_discount_rate'] = (df['base_price'] - df['checkout_price']) / df['base_price']
df.head()

,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area,new_discount_rate
0,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,0.101517
1,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,-0.007362
2,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,0.007361
3,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,0.224053
4,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,-0.004124


In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
df=df.drop("checkout_price",axis=1)

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
#Her haftada her bir mutfak türünden kaç farklı yemek sunulduğunu hesaplar
weekly_cuisine_category = df.groupby(['week', 'cuisine'])['category'].nunique().reset_index()
weekly_cuisine_category.rename(columns={'category': 'weekly_cuisine_cat'}, inplace=True)

# weekly_cuisine_cat sütununu df veri çerçevesine ekleyin
df = df.merge(weekly_cuisine_category, on=['week', 'cuisine'], how='left')
df.head()

,week,center_id,meal_id,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area,new_discount_rate,weekly_cuisine_cat
0,1,55,1885,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,0.101517,5
1,1,55,1993,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,-0.007362,5
2,1,55,2539,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,0.007361,5
3,1,55,2139,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,0.224053,4
4,1,55,2631,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,-0.004124,4


In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
#Her kategoride her bir mutfak türünün base_price'ını hesaplar.
cat_cuisine_price = df.groupby(['category', 'cuisine'])['base_price'].nunique().reset_index()
cat_cuisine_price.rename(columns={'base_price': 'cat_cuisine_price'}, inplace=True)

# cat_cuisine_price sütununu df veri çerçevesine ekleyin
df = df.merge(cat_cuisine_price, on=['category', 'cuisine'], how='left')
df.head()

,week,center_id,meal_id,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area,new_discount_rate,weekly_cuisine_cat,cat_cuisine_price
0,1,55,1885,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0,0.101517,5,233
1,1,55,1993,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0,-0.007362,5,233
2,1,55,2539,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0,0.007361,5,233
3,1,55,2139,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0,0.224053,4,473
4,1,55,2631,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0,-0.004124,4,473


In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
df['week'] = df['week'].astype(int)

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
df_train=df[df.week<=119]

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df_test=df[df.week>119]

In [16]:
# --- [CELL 15]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
df_train=df_train.drop("week",axis=1)
df_test=df_test.drop("week",axis=1)

In [17]:
# --- [CELL 16]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 17}
b=list(df_train.columns)

In [18]:
# --- [CELL 17]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 18}
df_encoded = pd.get_dummies(df_train[b], drop_first=True)

df_encoded.head()

,base_price,num_orders,op_area,new_discount_rate,weekly_cuisine_cat,cat_cuisine_price,center_id_11,center_id_13,center_id_14,center_id_17,...,city_code_713,region_code_34,region_code_35,region_code_56,region_code_71,region_code_77,region_code_85,region_code_93,center_type_TYPE_B,center_type_TYPE_C
0,152.29,177,2.0,0.101517,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
1,135.83,270,2.0,-0.007362,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
2,135.86,189,2.0,0.007361,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
3,437.53,54,2.0,0.224053,4,473,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
4,242.50,40,2.0,-0.004124,4,473,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True


In [19]:
# --- [CELL 18]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 19}
df_test_encoded=pd.get_dummies(df_test[b], drop_first=True)
df_test_encoded.head()

,base_price,num_orders,op_area,new_discount_rate,weekly_cuisine_cat,cat_cuisine_price,center_id_11,center_id_13,center_id_14,center_id_17,...,city_code_713,region_code_34,region_code_35,region_code_56,region_code_71,region_code_77,region_code_85,region_code_93,center_type_TYPE_B,center_type_TYPE_C
371136,151.38,148,2.0,0.000000,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
371137,152.35,123,2.0,0.006564,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
371138,150.35,136,2.0,0.083472,5,233,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
371139,307.49,13,2.0,0.000000,4,473,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
371140,155.26,27,2.0,0.000000,4,473,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True


In [20]:
# --- [CELL 19]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 20}
X = df_encoded.drop("num_orders", axis=1)##X_train
y = df_encoded["num_orders"]##y_train

In [21]:
# --- [CELL 20]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 21}
w=df_test_encoded.drop("num_orders", axis=1)##X_test
z=df_test_encoded["num_orders"]#y_test

In [22]:
# --- [CELL 21]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 22}
from sklearn.preprocessing import StandardScaler

In [23]:
# --- [CELL 22]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 23}
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from math import sqrt

# Decision Tree Regressor için parametre gridini tanımlayın
param_grid = {
    'max_depth': [10, 15, 20],  # Farklı maksimum derinlik seviyeleri
    'min_samples_split': [2, 5],  # Farklı min_samples_split değerleri
    'min_samples_leaf': [1, 2]  # Farklı min_samples_leaf değerleri
}

# Decision Tree Regressor modelini oluşturun
DTRmodel = DecisionTreeRegressor(random_state=0)

# GridSearchCV'yi tanımlayın
grid_search = GridSearchCV(estimator=DTRmodel, param_grid=param_grid, 
                           cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Modeli eğitim verileri üzerinde eğitin
grid_search.fit(X, y)

# En iyi parametreleri ve en iyi tahmin modelini alın
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_

# En iyi tahmin modelini kullanarak test verileri üzerinde tahmin yapın
y_pred = best_estimator.predict(w)

# Performans metriklerini hesaplayın
r2 = r2_score(z, y_pred)
mse = mean_squared_error(z, y_pred)
rmse = sqrt(mse)

# Sonuçları yazdırın
print("En iyi parametreler:", best_params)
print("R2 score:", r2)
print("MSE score:", mse)
print("RMSE:", rmse)

En iyi parametreler: {'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 5}
R2 score: 0.6792665676667602
MSE score: 42782.72208333984
RMSE: 206.83984645937986


In [24]:
# --- [CELL 23]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 24}
dft['new_discount_rate'] = (dft['base_price'] - dft['checkout_price']) / dft['base_price']

In [25]:
# --- [CELL 24]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 25}
dft=dft.drop("checkout_price",axis=1)

In [26]:
# --- [CELL 25]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 26}
#Her haftada her bir mutfak türünden kaç farklı yemek sunulduğunu hesaplar
weekly_cuisine_category = dft.groupby(['week', 'cuisine'])['category'].nunique().reset_index()
weekly_cuisine_category.rename(columns={'category': 'weekly_cuisine_cat'}, inplace=True)

# weekly_cuisine_cat sütununu df veri çerçevesine ekleyin
dft = dft.merge(weekly_cuisine_category, on=['week', 'cuisine'], how='left')
dft.head()

,id,week,center_id,meal_id,base_price,emailer_for_promotion,homepage_featured,category,cuisine,city_code,region_code,center_type,op_area,new_discount_rate,weekly_cuisine_cat
0,1028232,146,55,1885,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0,0.006285,5
1,1127204,146,55,1993,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0,-0.006285,5
2,1212707,146,55,2539,159.14,0,0,Beverages,Thai,647,56,TYPE_C,2.0,0.012568,5
3,1082698,146,55,2631,162.02,0,0,Beverages,Indian,647,56,TYPE_C,2.0,0.000000,4
4,1400926,146,55,1248,163.93,0,0,Beverages,Indian,647,56,TYPE_C,2.0,0.000000,4


In [27]:
# --- [CELL 26]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 27}
#Her kategoride her bir mutfak türünün base_price'ını hesaplar.
cat_cuisine_price = dft.groupby(['category', 'cuisine'])['base_price'].nunique().reset_index()
cat_cuisine_price.rename(columns={'base_price': 'cat_cuisine_price'}, inplace=True)

# cat_cuisine_price sütununu df veri çerçevesine ekleyin
dft = dft.merge(cat_cuisine_price, on=['category', 'cuisine'], how='left')
dft.head()

,id,week,center_id,meal_id,base_price,emailer_for_promotion,homepage_featured,category,cuisine,city_code,region_code,center_type,op_area,new_discount_rate,weekly_cuisine_cat,cat_cuisine_price
0,1028232,146,55,1885,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0,0.006285,5,101
1,1127204,146,55,1993,159.11,0,0,Beverages,Thai,647,56,TYPE_C,2.0,-0.006285,5,101
2,1212707,146,55,2539,159.14,0,0,Beverages,Thai,647,56,TYPE_C,2.0,0.012568,5,101
3,1082698,146,55,2631,162.02,0,0,Beverages,Indian,647,56,TYPE_C,2.0,0.000000,4,183
4,1400926,146,55,1248,163.93,0,0,Beverages,Indian,647,56,TYPE_C,2.0,0.000000,4,183


In [28]:
# --- [CELL 27]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 28}
dft=dft.drop("id",axis=1)

In [29]:
# --- [CELL 28]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 29}
dft=dft.drop('week',axis=1)

In [30]:
# --- [CELL 29]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 30}
c = list(dft.columns)
c

['center_id',
 'meal_id',
 'base_price',
 'emailer_for_promotion',
 'homepage_featured',
 'category',
 'cuisine',
 'city_code',
 'region_code',
 'center_type',
 'op_area',
 'new_discount_rate',
 'weekly_cuisine_cat',
 'cat_cuisine_price']

In [31]:
# --- [CELL 30]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 31}
dft_encoded = pd.get_dummies(dft[c], drop_first=True)
dft_encoded.head()

,base_price,op_area,new_discount_rate,weekly_cuisine_cat,cat_cuisine_price,center_id_11,center_id_13,center_id_14,center_id_17,center_id_20,...,city_code_713,region_code_34,region_code_35,region_code_56,region_code_71,region_code_77,region_code_85,region_code_93,center_type_TYPE_B,center_type_TYPE_C
0,159.11,2.0,0.006285,5,101,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
1,159.11,2.0,-0.006285,5,101,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
2,159.14,2.0,0.012568,5,101,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
3,162.02,2.0,0.000000,4,183,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
4,163.93,2.0,0.000000,4,183,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True


In [32]:
# --- [CELL 31]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 32}
# === BEFORE (original) ===
# final_pred = DTRmodel.predict(dft_encoded)

# === AFTER (edited) ===
final_pred = best_estimator.predict(dft_encoded)

In [33]:
import numpy as np

assert "final_pred" in globals(), "final_pred should be defined by the previous cell"
expected_pred = best_estimator.predict(dft_encoded)
assert final_pred.shape == expected_pred.shape, "Prediction shape should match best_estimator output"
assert np.array_equal(final_pred, expected_pred), "final_pred must be computed using best_estimator.predict(dft_encoded)"